In [1]:
import pandas as pd

def load_data(file_path):
    data = pd.read_csv(file_path, sep=';', header=None, names=['Text', 'Emotion'])
    return data

In [2]:
train_df = load_data('archive-3/train.txt')
val_df = load_data('archive-3/val.txt')
test_df = load_data('archive-3/test.txt')

In [3]:
print("Train shape:", train_df.shape)


Train shape: (16000, 2)


In [5]:
train_df.head()

,Text,Emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [7]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/saurabh/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /Users/saurabh/nltk_data...


True

In [8]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(cleaned_words)

train_df['Cleaned_Text'] = train_df['Text'].apply(clean_text)
val_df['Cleaned_Text'] = val_df['Text'].apply(clean_text)
test_df['Cleaned_Text'] = test_df['Text'].apply(clean_text)

In [9]:
print(train_df[['Text', 'Cleaned_Text']].head())

                                                Text  \
0                            i didnt feel humiliated   
1  i can go from feeling so hopeless to so damned...   
2   im grabbing a minute to post i feel greedy wrong   
3  i am ever feeling nostalgic about the fireplac...   
4                               i am feeling grouchy   

                                        Cleaned_Text  
0                              didnt feel humiliated  
1  go feeling hopeless damned hopeful around some...  
2          im grabbing minute post feel greedy wrong  
3  ever feeling nostalgic fireplace know still pr...  
4                                    feeling grouchy  


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train = tfidf.fit_transform(train_df['Cleaned_Text'])
X_val = tfidf.transform(val_df['Cleaned_Text'])
X_test = tfidf.transform(test_df['Cleaned_Text'])

y_train = train_df['Emotion']
y_val = val_df['Emotion']
y_test = test_df['Emotion']

print("X_train shape:", X_train.shape)

X_train shape: (16000, 5000)


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

trained_models = {}

for name, model in models.items():
    print(f"--- Training {name} ---")
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_val_pred)
    print(f"{name} Validation Accuracy: {acc:.4f}\n")
    trained_models[name] = model

--- Training Logistic Regression ---
Logistic Regression Validation Accuracy: 0.8760

--- Training Decision Tree ---
Decision Tree Validation Accuracy: 0.8780

--- Training Random Forest ---
Random Forest Validation Accuracy: 0.8935



In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear']
}

grid_search = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Hyperparameters found:", grid_search.best_params_)
best_model = grid_search.best_estimator_

tuned_val_acc = accuracy_score(y_val, best_model.predict(X_val))
print(f"Tuned Logistic Regression Validation Accuracy: {tuned_val_acc:.4f}")

/Users/saurabh/Documents/NLP_RNN_Assignmenet/myenv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:493: FitFailedWarning: 
9 fits failed out of a total of 18.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/saurabh/Documents/NLP_RNN_Assignmenet/myenv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/saurabh/Documents/NLP_RNN_Assignmenet/myenv/lib/python3.12/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/U

Best Hyperparameters found: {'C': 10, 'solver': 'lbfgs'}
Tuned Logistic Regression Validation Accuracy: 0.8905


In [14]:
from sklearn.metrics import f1_score

models["Tuned Logistic Regression"] = best_model
evaluation_results = []

for name, model in models.items():
    y_test_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred, average='weighted')

    evaluation_results.append({
        "Model": name,
        "Accuracy": round(acc * 100, 2),
        "F1-Score": round(f1 * 100, 2)
    })

results_df = pd.DataFrame(evaluation_results)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("\n=== Model Performance Comparison Table ===")
display(results_df)

best_row = results_df.iloc[0]
print(f"\n Suggested Best Model: {best_row['Model']} with an Accuracy of {best_row['Accuracy']}% and F1-Score of {best_row['F1-Score']}%.")


=== Model Performance Comparison Table ===


,Model,Accuracy,F1-Score
0,Random Forest,88.55,88.49
1,Tuned Logistic Regression,88.45,88.32
2,Logistic Regression,87.50,87.10
3,Decision Tree,86.25,86.32



 Suggested Best Model: Random Forest with an Accuracy of 88.55% and F1-Score of 88.49%.
